# SCNN v8 — FHT + ECA + Expanded Training + Better Calibration

Changes: all sessions from non-interday train subjects, circular aug 4x, 50 epochs, LogReg C=0.1.

In [1]:
import sys, math
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.linear_model import LogisticRegression

from config import RANDOM_SEED, N_CLASSES, MODELS_DIR, get_device
from src.experiment_runner import (
    get_splits, load_and_norm,
    run_zero_shot, run_calibration, print_comparison,
)
from src.feature_extraction import fht_envelope_batch
from src.evaluation import measure_latency, print_latency

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
DEVICE = get_device()
splits = get_splits()
print(f'Device: {DEVICE}')

Device: mps


In [2]:
class ECA2d(nn.Module):
    def __init__(self,ch):
        super().__init__()
        k = max(int(abs(math.log2(ch)/2+0.5)),3); k = k if k%2 else k+1
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1,1,k,padding=k//2,bias=False)
    def forward(self,x):
        b,c,h,w = x.size()
        return x * torch.sigmoid(self.conv(self.gap(x).view(b,1,c))).view(b,c,1,1).expand_as(x)

class DSConv2d(nn.Module):
    def __init__(self,ic,oc,k=3,p=1):
        super().__init__()
        self.dw = nn.Conv2d(ic,ic,k,padding=p,groups=ic)
        self.pw = nn.Conv2d(ic,oc,1)
    def forward(self,x): return self.pw(self.dw(x))

class SCNN_ECA(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            DSConv2d(1,48), nn.BatchNorm2d(48), nn.ReLU(), ECA2d(48), nn.MaxPool2d(2),
            DSConv2d(48,96), nn.BatchNorm2d(96), nn.ReLU(), ECA2d(96), nn.MaxPool2d(2),
            DSConv2d(96,192), nn.BatchNorm2d(192), nn.ReLU(), ECA2d(192), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(192,96), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(96,n_classes),
        )
    def forward(self,x): return self.classifier(self.features(x))
    def extract_feat(self,x):
        with torch.no_grad(): return nn.Flatten()(self.features(x))

print(f'Params: {sum(p.numel() for p in SCNN_ECA().parameters()):,}')

Params: 44,764


In [3]:
def circular_augment(X, y, n_rot=4):
    N,C,T = X.shape
    Xa = np.empty((N*n_rot,C,T), dtype=X.dtype)
    ya = np.empty(N*n_rot, dtype=y.dtype)
    for s in range(n_rot):
        Xa[s*N:(s+1)*N] = np.roll(X, shift=s*2, axis=1)  # shifts: 0,2,4,6
        ya[s*N:(s+1)*N] = y
    idx = np.random.RandomState(RANDOM_SEED).permutation(len(ya))
    return Xa[idx], ya[idx]

class NoisyDS(Dataset):
    def __init__(self,X,y,std=0.1):
        self.X = torch.from_numpy(X).float().unsqueeze(1)
        self.y = torch.from_numpy(y).long(); self.std = std
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i]+torch.randn_like(self.X[i])*self.std, self.y[i]

In [4]:
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Raw train: {X_train.shape}')

X_fht = fht_envelope_batch(X_train)
X_aug, y_aug = circular_augment(X_fht, y_train)
print(f'After FHT + circ aug: {X_aug.shape[0]:,}')

model = SCNN_ECA().to(DEVICE)
loader = DataLoader(NoisyDS(X_aug, y_aug), batch_size=256, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=50, steps_per_epoch=len(loader))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

for ep in range(50):
    model.train()
    ls,c,t = 0,0,0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE),yb.to(DEVICE)
        out = model(xb); loss = crit(out,yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        ls += loss.item()*xb.size(0); c += (out.argmax(1)==yb).sum().item(); t += xb.size(0)
    if (ep+1)%10==0 or ep==0:
        print(f'Epoch {ep+1:2d}/50 — loss:{ls/t:.4f} acc:{c/t:.4f}')

torch.save(model.state_dict(), MODELS_DIR / 'scnn_v8.pt')
print('Saved.')

Loading windows: 100%|██████████| 9021/9021 [00:04<00:00, 2082.10it/s]


Raw train: (1030712, 8, 50)
After FHT + circ aug: 4,122,848
Epoch  1/50 — loss:1.5146 acc:0.4684
Epoch 10/50 — loss:1.3329 acc:0.5688
Epoch 20/50 — loss:1.3339 acc:0.5681
Epoch 30/50 — loss:1.3094 acc:0.5802
Epoch 40/50 — loss:1.2728 acc:0.5993
Epoch 50/50 — loss:1.2457 acc:0.6134
Saved.


In [5]:
@torch.no_grad()
def scnn_predict(X):
    model.eval()
    Xf = fht_envelope_batch(X)
    Xt = torch.from_numpy(Xf).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model(xb[0].to(DEVICE)).argmax(1).cpu().numpy() for xb in loader])

@torch.no_grad()
def scnn_features(X):
    model.eval()
    Xf = fht_envelope_batch(X)
    Xt = torch.from_numpy(Xf).float().unsqueeze(1)
    loader = DataLoader(TensorDataset(Xt), batch_size=512, shuffle=False)
    return np.concatenate([model.extract_feat(xb[0].to(DEVICE)).cpu().numpy() for xb in loader])

def scnn_finetune(X_cal, y_cal):
    F = scnn_features(X_cal)
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=0.1)
    clf.fit(F, y_cal)
    def predict_ft(X): return clf.predict(scnn_features(X))
    return predict_ft

print('Option B — Zero-shot:')
zero_results = run_zero_shot(scnn_predict, splits, norm_stats)
print('\nOption A — Calibration:')
cal_results = run_calibration(scnn_predict, scnn_finetune, splits, norm_stats)

Option B — Zero-shot:
  S1 zero-shot: 0.5440
  S2 zero-shot: 0.5168
  S3 zero-shot: 0.5193
  S4 zero-shot: 0.5825
  S5 zero-shot: 0.6686

Option A — Calibration:
  S1 calibrated: 0.7128
  S2 calibrated: 0.7014
  S3 calibrated: 0.7637
  S4 calibrated: 0.7268
  S5 calibrated: 0.7922


In [6]:
model.eval()
sample = X_train[:1]
sf = torch.from_numpy(fht_envelope_batch(sample)).float().unsqueeze(1).to(DEVICE)
for _ in range(10): _ = model(sf)
if DEVICE.type=='mps': torch.mps.synchronize()
def scnn_single(x):
    xf = fht_envelope_batch(x)
    xt = torch.from_numpy(xf).float().unsqueeze(1).to(DEVICE)
    with torch.no_grad(): out = model(xt)
    if DEVICE.type=='mps': torch.mps.synchronize()
    return out.argmax(1).cpu().numpy()
latency = measure_latency(scnn_single, sample, n_runs=500)
print_latency(latency, 'SCNN+FHT')
print_comparison(zero_results, cal_results, name='SCNN+FHT (v8)')


Latency — SCNN+FHT
  Mean:   1.53 ms
  Median: 1.34 ms
  P95:    2.09 ms
  <300ms: ✓

  SCNN+FHT (v8) — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                54.40%       71.28%   +16.88%
S2                51.68%       70.14%   +18.46%
S3                51.93%       76.37%   +24.44%
S4                58.25%       72.68%   +14.43%
S5                66.86%       79.22%   +12.37%
